# Transformação de Dados da Tabela "Natureza de Ocupação"

## Configuração do Ambiente

In [0]:
from pyspark.sql.utils import AnalysisException
from src.utils.udfs import functions_for_df_structure_management as ffdsm

## Ingestão de dados da camada de bronze

In [0]:
df_nature_of_occupation = spark.table("fiap_1ctor.bronze_layer.delta_natureza_de_ocupacao")

In [0]:
display(df_nature_of_occupation)

## Transformação de Dados

In [0]:
df_nature_of_occupation = df_nature_of_occupation.dropna(how='all')

In [0]:
df_casted_nature_of_occupation = ffdsm.cast_columns_to_float(df_nature_of_occupation, ["AnoCalendario", "NaturezaDeOcupacao"])

In [0]:
df_casted_nature_of_occupation = ffdsm.rename_columns_with_df_name(df_casted_nature_of_occupation, "NaturezaDeOcupacao", ["AnoCalendario", "NaturezaDeOcupacao"])

In [0]:
dbutils.data.summarize(df_casted_nature_of_occupation)

In [0]:
df_filled_nature_of_occupation = ffdsm.fill_nulls(df_casted_nature_of_occupation, ["AnoCalendario", "NaturezaDeOcupacao"])

In [0]:
df_nulls = ffdsm.count_nulls(df_filled_nature_of_occupation)
display(df_nulls)

## Salvar como Delta na Camada Silver

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS fiap_1ctor.silver_layer")

In [0]:
error = None

try:
    df_filled_nature_of_occupation.write \
        .mode("overwrite") \
        .saveAsTable(f"fiap_1ctor.silver_layer.delta_natureza_de_ocupacao")
    error = None
except Exception as e:
    error = str(e)
    print(error)